# Dataset 5 — Inspection
**German adolescents, three-group RCT (control / default friction / reintervention)**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW5 = Path("../data_raw/5. data set, German adolescents")
BEH5  = RAW5 / "Behavior data set/behavior_data_set_5.csv"
PRE5  = RAW5 / "Survey data sets/Pre-survey/pre_survey_data_set_5.csv"
POST5 = RAW5 / "Survey data sets/Post-survey/post_survey_data_set_5.csv"
print("Paths configured")

## 1. File Inventory

In [ ]:
import os
for root, dirs, files in os.walk(RAW5):
    level = Path(root).relative_to(RAW5)
    for f in files:
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1e6
        print(f"  {level}/{f}  ({size_mb:.1f} MB)")

## 2. Behavioral File — Structure

In [ ]:
beh = pd.read_csv(BEH5, sep=';', low_memory=False)
print("Shape:", beh.shape)
print("\nColumns:", beh.columns.tolist())
beh.head(3)

## 3. Delimiter Confirmation

In [ ]:
with open(BEH5) as f:
    header = f.readline()
print("Header (first 200 chars):", header[:200])
print("Semicolons in header:", header.count(';'))
print("Commas in header:", header.count(','))

## 4. Timestamp Format

In [ ]:
print("Timestamp samples:")
print(beh['timestamp'].head(5).tolist())
parsed = pd.to_datetime(beh['timestamp'], errors='coerce')
print("\nParsed (naive):", parsed.head(3).tolist())
print("Parse failures:", parsed.isna().sum())

## 5. Timezone Column

In [ ]:
print("Unique timezones:", beh['timezone'].nunique())
print(beh['timezone'].value_counts(dropna=False).head(10))

## 6. Resolution (Event Outcome)

In [ ]:
print(beh['resolution'].value_counts(dropna=False))

## 7. Participant IDs — userid

In [ ]:
print("Unique userids:", beh['userid'].nunique())
print("\nSamples:")
for uid in beh['userid'].head(3):
    print(" ", uid)

## 8. ID Reconciliation: userid vs uuid

In [ ]:
PREFIX = "study-one-sec-next-gen-bzkj-2024-"
beh['userid_clean'] = beh['userid'].str.removeprefix(PREFIX)

pre = pd.read_csv(PRE5, sep=';', low_memory=False)
post = pd.read_csv(POST5, sep=';', low_memory=False)

beh_ids = set(beh['userid_clean'].unique())
pre_ids = set(pre['uuid'].unique())
post_ids = set(post['uuid'].unique())

print(f"Behavioral IDs (after strip): {len(beh_ids)}")
print(f"Pre-survey UUIDs: {len(pre_ids)}")
print(f"Post-survey UUIDs: {len(post_ids)}")
print(f"Beh ∩ Pre: {len(beh_ids & pre_ids)}")
print(f"Beh ∩ Post: {len(beh_ids & post_ids)}")
print(f"Pre ∩ Post: {len(pre_ids & post_ids)}")
print(f"All three: {len(beh_ids & pre_ids & post_ids)}")

## 9. Condition Column — Not in Behavioral

In [ ]:
print("'cond' in behavioral:", 'cond' in beh.columns)
print("\nCond in pre-survey:")
print(pre['cond'].value_counts(dropna=False))

## 10. Reason Column (German free text)

In [ ]:
print(f"Reason NaN: {beh['reason'].isna().sum()} ({beh['reason'].isna().mean()*100:.1f}%)")
print("\nTop 15 reason values:")
print(beh['reason'].value_counts(dropna=False).head(15))

## 11. Intervention Duration and Re-intervention

In [ ]:
print("interventionDuration top values:")
print(beh['interventionDuration'].value_counts(dropna=False).head(8))
print()
print("timeIntervalUntilReIntervention top values:")
print(beh['timeIntervalUntilReIntervention'].value_counts(dropna=False).head(8))

## 12. Suggestion Columns

In [ ]:
sug_cols = ['suggestionBannerShown','suggestionDetailViewAppear',
            'suggestionDetailViewDisappear','terminationReason','suggestionLearnMoreLinkTapped']
for col in sug_cols:
    n = beh[col].notna().sum()
    print(f"{col}: {n} non-null ({n/len(beh)*100:.2f}%)")

## 13. Full Null Summary — Behavioral

In [ ]:
null_pct = (beh.isnull().sum() / len(beh) * 100).round(1)
pd.DataFrame({'null_count': beh.isnull().sum(), 'null_pct': null_pct})

## 14. Pre-Survey Structure

In [ ]:
print("Shape:", pre.shape)
print("Columns:", pre.columns.tolist())
print("\nNull counts:")
print(pre.isnull().sum())

## 15. Pre-Survey Scale Distributions

In [ ]:
scale_means = {}
for scale, cols in [
    ('belong', [f'belong_{i}' for i in range(1,9)]),
    ('ostracism', [f'ostracism_{i}' for i in range(1,6)]),
    ('fomo', [f'fomo_{i}' for i in range(1,11)]),
]:
    for col in cols:
        pre[col] = pd.to_numeric(pre[col], errors='coerce')
    scale_means[scale] = pre[cols].mean(axis=1)
    print(f"{scale} mean: {scale_means[scale].mean():.2f} (sd={scale_means[scale].std():.2f}, NaN={scale_means[scale].isna().sum()})")

## 16. Post-Survey Structure

In [ ]:
print("Shape:", post.shape)
print("Columns:", post.columns.tolist())
print("\nUnique UUIDs:", post['uuid'].nunique())
print("Duplicate UUIDs:", post.duplicated(subset=['uuid']).sum())

## 17. Age Distribution

In [ ]:
print(pre['age'].value_counts(dropna=False).sort_index())

## 18. Duplicate Rows Check

In [ ]:
for name, df in [('behavioral', beh), ('pre-survey', pre), ('post-survey', post)]:
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

## 19. Inspection Summary

In [ ]:
print("""
=== DS5 INSPECTION SUMMARY ===

Behavioral (288,688 rows × 13 cols):
  Delimiter: semicolon
  Timestamp: local ISO string → localize via timezone col → UTC
  userid prefix: 'study-one-sec-next-gen-bzkj-2024-' must be stripped
  573 unique participants
  Resolution: openedApp 61.6%, dismissedAppOpening 21.5%, closedApp 15.6%, NaN 1.2%, dismissedByStructuredIntervention 0.04%
  reason: 42.3% NaN (German free text)
  interventionDuration: 0 for opened, 6 for dismissed, NaN for 48,742
  timeIntervalUntilReIntervention: 77.9% NaN (reintervention group only)
  suggestion columns: 99%+ NaN
  terminationReason: 12 non-null rows

Pre-survey (792 rows × 37 cols):
  792 unique UUIDs, 0 duplicates
  3 conditions: control=265, default=264, reintervention=263
  Scales: belong (8 items), ostracism (5 items), fomo (10 items)

Post-survey (314 rows → 311 after dedup):
  311 unique UUIDs, 3 duplicate UUIDs (first kept)
  Additional columns: solve_prob, address_goal

ID overlap:
  Beh ∩ Pre: 572 | Beh ∩ Post: 284 | All 3: 284

Ambiguities:
  - cond not in behavioral — join from pre-survey
  - reason is German text — not categorized
  - dismissedByStructuredIntervention (119 rows) is undocumented
  - 220 pre-survey participants have no behavioral data
""")